# 03. Advanced Aggregation in Pandas

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week10/03.Advanced-Aggregation/notebooks/01_03.Advanced-Aggregation.ipynb)

## Overview
While basic GroupBy operations compute a single statistic across all columns, real-world reporting demands **multi-metric summaries** (e.g. mean, standard deviation, and count) and **column-specific aggregations** (e.g. summing revenue while averaging customer ratings).

In this notebook, we cover:
1. **Multi-Metric Aggregation**: Passing lists of statistical functions to `.agg()`.
2. **Dictionary Column Mappings**: Applying distinct functions to specific columns.
3. **Modern Named Aggregations**: Creating clean, flat column names directly during aggregation (`new_name=('col', 'func')`).

## 1. Setup: Faculty Research Grants and Publications

Consider faculty-level performance metrics across Australian university campuses.

In [ ]:
import pandas as pd
import numpy as np

df_research = pd.DataFrame({
    'faculty': [
        'Health Sciences', 'Health Sciences', 'Health Sciences',
        'Education & Arts', 'Education & Arts', 'Education & Arts',
        'Law & Business', 'Law & Business', 'Law & Business',
        'Theology & Philosophy', 'Theology & Philosophy', 'Theology & Philosophy'
    ],
    'campus': [
        'North Sydney', 'Melbourne', 'Brisbane',
        'North Sydney', 'Melbourne', 'Strathfield',
        'North Sydney', 'Melbourne', 'Brisbane',
        'North Sydney', 'Melbourne', 'Brisbane'
    ],
    'research_grant_kaud': [450, 520, 380, 210, 240, 190, 310, 340, 280, 150, 180, 130],
    'publications': [28, 34, 22, 15, 18, 12, 19, 21, 16, 11, 14, 9],
    'satisfaction_pct': [89.5, 91.0, 87.5, 92.0, 93.5, 90.0, 85.0, 86.5, 84.0, 94.0, 95.5, 93.0]
})

print("Faculty Research Dataset:")
display(df_research)

## 2. Multi-Metric Aggregation on a Single Column

Pass a list of function names to `.agg()` to compute comprehensive summary statistics simultaneously.

In [ ]:
grant_summary = df_research.groupby('faculty')['research_grant_kaud'].agg(
    ['count', 'mean', 'std', 'min', 'max']
)
print("Research Grant Metrics by Faculty:")
display(grant_summary.round(1))

## 3. Column-Specific Aggregation with Dictionaries

Pass a dictionary to `.agg()` to map each column to its intended statistical function(s).

In [ ]:
agg_mapping = {
    'research_grant_kaud': ['sum', 'mean'],
    'publications': 'sum',
    'satisfaction_pct': 'mean'
}

campus_summary = df_research.groupby('campus').agg(agg_mapping)
print("Campus Summary (Dictionary Mapping):")
display(campus_summary.round(1))

## 4. Modern Named Aggregations (Flat Column Layout)

Dictionary aggregations with multiple functions create hierarchical MultiIndex columns (e.g. `('research_grant_kaud', 'sum')`).

Pandas **Named Aggregation** syntax avoids this by allowing you to define flat column names during aggregation:
```python
df.groupby(...).agg(
    new_column_name=('target_column', 'function')
)
```

In [ ]:
clean_named = df_research.groupby('campus').agg(
    total_projects=('research_grant_kaud', 'count'),
    total_grants_kaud=('research_grant_kaud', 'sum'),
    avg_grant_kaud=('research_grant_kaud', 'mean'),
    total_pubs=('publications', 'sum'),
    mean_satisfaction=('satisfaction_pct', 'mean')
)

print("Clean Summary with Flat Column Names:")
display(clean_named.round(1))
print(f"\nFlat column headers: {clean_named.columns.tolist()}")

## 5. Practical Exercises

### Exercise 1: Multi-Metric Energy Consumption by Sector
Using the Australian energy dataset below:
1. Group by `sector`.
2. Compute the `min`, `mean`, and `max` of `energy_gwh`.

In [ ]:
energy = pd.DataFrame({
    'state': ['NSW', 'NSW', 'VIC', 'VIC', 'QLD', 'QLD', 'WA', 'WA'],
    'sector': ['Residential', 'Commercial', 'Residential', 'Commercial', 'Residential', 'Commercial', 'Residential', 'Commercial'],
    'energy_gwh': [4200, 3100, 3800, 2900, 3500, 2700, 2100, 1800],
    'cost_maud': [630.0, 465.0, 570.0, 435.0, 490.0, 378.0, 294.0, 252.0]
})

# --- Student Code Here ---
# sector_stats = ...

# --- Solution ---
sector_stats = energy.groupby('sector')['energy_gwh'].agg(['min', 'mean', 'max'])
display(sector_stats)

### Exercise 2: State Named Aggregation
Group `energy` by `state` using Named Aggregation to compute:
- `total_energy`: sum of `energy_gwh`
- `avg_cost`: mean of `cost_maud`

In [ ]:
# --- Student Code Here ---
# state_stats = ...

# --- Solution ---
state_stats = energy.groupby('state').agg(
    total_energy=('energy_gwh', 'sum'),
    avg_cost=('cost_maud', 'mean')
)
display(state_stats.round(1))

## 6. Key Takeaways

1. **`.agg(['func1', 'func2'])`**: Computes multiple descriptive statistics across groups simultaneously.
2. **Dictionary Aggregation**: Tailors specific aggregators to individual columns.
3. **Named Aggregation (`new_col=('target', 'func')`)**: The cleanest modern approach, producing flat, publication-ready column headers.